In [ ]:
%load_ext autoreload
%autoreload 2

# Otimização Optuna

## Inicialização dos datasets

In [ ]:
from src.Anomaly.Optimizer import AnomalyOptunaOptimizer
from src.Data.Processor import DataStreamProcessor
import pandas as pd
from datetime import datetime

# Definição dos datasets
categorias = ['Consistência', 'Generalização', 'Adaptação', 'Recorrência']
tamanhos = ['25', '200', '1000']

datasets = [f'data/15k/{cat}/{cat}_{tam}.csv' for cat in categorias for tam in tamanhos]

FEATURES = [
    'Max Packet Length',
    'Average Packet Size',
    'Fwd Packet Length Min',
    'Min Packet Length',
    'Fwd Packet Length Max',
    'Packet Length Mean',
    'Fwd Packet Length Mean',
    'Avg Fwd Segment Size',
    'min_seg_size_forward',
    'ACK Flag Count',
    'Flow Duration',
    'Fwd IAT Total',
    'Flow IAT Max',
    'Fwd IAT Max',
    'Flow IAT Std',
    'Fwd IAT Std',
    'Fwd IAT Mean',
    'Flow IAT Mean',
    'Total Length of Fwd Packets',
    'Subflow Fwd Bytes',
    'act_data_pkt_fwd',
    'Subflow Fwd Packets',
    'Total Fwd Packets',
    'Down/Up Ratio',
    'Init_Win_bytes_backward',
    'Total Length of Bwd Packets',
    'Subflow Bwd Bytes',
    'Flow IAT Min',
    'Bwd Packet Length Max',
    'URG Flag Count',
    'Bwd IAT Total',
    'Bwd Packets/s',
    'Init_Win_bytes_forward',
]

In [ ]:
from datetime import datetime
import pandas as pd

id_execucao = datetime.now().strftime("%Y%m%d_%H%M")

modelos_para_executar = ['AE', 'HST', 'AIF']
estrategias_discretizacao = ['z_score', 'dinamic']
configuracoes_features = [('FullFeatures', None), ('33Features', FEATURES)]

for dataset_path in datasets:
    nome_experimento = dataset_path.split('/')[-1].replace('.csv', '')
    df = pd.read_csv(dataset_path)
    
    for feat_name, selected_feats in configuracoes_features:
        processor = DataStreamProcessor(logging=False, selected_features=selected_feats)
        
        stream, targets, features = processor.create_stream(
            df=df,
            target_label_col='Label',
            binary_label=False,
            normalize_method="MinMaxScaler",
            threshold_var=None,
            threshold_corr=None,
            top_n_features=None,
            return_stream=True,
            extra_ignore_cols=['Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Protocol', 'Inbound'],
            imputation_method='mediana'
        )
        
        for estrategia in estrategias_discretizacao:        
            for model_name in modelos_para_executar:
                print(f"\n---> Otimizando | Modelo: {model_name} | Discretização: {estrategia} | Feats: {feat_name} <---")
                
                optimizer = AnomalyOptunaOptimizer(
                    stream=stream,
                    n_trials=75,
                    discretization_threshold=estrategia,
                    target_names=targets,
                    n_runs=5
                )
                
                best_params = optimizer.optimize(
                    model_name=model_name,
                    warmup_instances=2000,
                    experiment_name=nome_experimento,
                    num_features=len(features),
                    exec_id=id_execucao,
                    window_evaluation=100
                )

## Todas as características

In [ ]:
from datetime import datetime
import pandas as pd

id_execucao = datetime.now().strftime("%Y%m%d_%H%M")

modelos_para_executar = ['AE', 'HST', 'AIF']
estrategias_discretizacao = ['z_score', 'dinamic']

for dataset_path in datasets:
    nome_experimento = dataset_path.split('/')[-1].replace('.csv', '')
    print(f"\n{'='*60}\nIniciando dataset: {nome_experimento}\n{'='*60}")
    
    df = pd.read_csv(dataset_path)
    
    processor = DataStreamProcessor(logging=False)
    
    stream, targets, features = processor.create_stream(
        df=df,
        target_label_col='Label',
        binary_label=False,
        normalize_method="MinMaxScaler",
        threshold_var=None,
        threshold_corr=None,
        top_n_features=None,
        return_stream=True,
        extra_ignore_cols=['Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Protocol', 'Inbound'],
        imputation_method='mediana'
    )
    
    for estrategia in estrategias_discretizacao:        
        for model_name in modelos_para_executar:
            print(f"\n---> Otimizando | Modelo: {model_name} | Discretização: {estrategia} <---")
            
            optimizer = AnomalyOptunaOptimizer(
                stream=stream,
                n_trials=75,
                discretization_threshold=estrategia,
                target_names=targets,
                n_runs=5
            )
            
            best_params = optimizer.optimize(
                model_name=model_name,
                warmup_instances=2000,
                experiment_name=nome_experimento,
                num_features=len(features),
                exec_id=id_execucao,
                window_evaluation=100
            )

## Melhores características

In [ ]:
from datetime import datetime
import pandas as pd

id_execucao = datetime.now().strftime("%Y%m%d_%H%M")

modelos_para_executar = ['AE', 'HST', 'AIF']
estrategias_discretizacao = ['z_score', 'dinamic']

for dataset_path in datasets:
    nome_experimento = dataset_path.split('/')[-1].replace('.csv', '')
    print(f"\n{'='*60}\nIniciando dataset: {nome_experimento}\n{'='*60}")
    
    df = pd.read_csv(dataset_path)
    
    processor = DataStreamProcessor(logging=False, selected_features=FEATURES)
    
    stream, targets, features = processor.create_stream(
        df=df,
        target_label_col='Label',
        binary_label=False,
        normalize_method="MinMaxScaler",
        threshold_var=None,
        threshold_corr=None,
        top_n_features=None,
        return_stream=True,
        extra_ignore_cols=['Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Protocol', 'Inbound'],
        imputation_method='mediana'
    )
    
    for estrategia in estrategias_discretizacao:        
        for model_name in modelos_para_executar:
            print(f"\n---> Otimizando | Modelo: {model_name} | Discretização: {estrategia} <---")
            
            optimizer = AnomalyOptunaOptimizer(
                stream=stream,
                n_trials=75,
                discretization_threshold=estrategia,
                target_names=targets,
                n_runs=5
            )
            
            best_params = optimizer.optimize(
                model_name=model_name,
                warmup_instances=2000,
                experiment_name=nome_experimento,
                num_features=len(features),
                exec_id=id_execucao,
                window_evaluation=100
            )

# Execução Default

## Inicialização dos Datasets

In [ ]:
from src.Anomaly.Pipeline import AnomalyExperimentRunner
from src.Anomaly.Models import get_anomaly_models
from src.Data.Processor import DataStreamProcessor
import pandas as pd

categorias = ['Consistência', 'Generalização', 'Adaptação', 'Recorrência']
tamanhos = ['25', '200', '1000']

# categorias = ['Generalização', 'Adaptação']
# tamanhos = ['200']

datasets = [f'data/15k/{cat}/{cat}_{tam}.csv' for cat in categorias for tam in tamanhos]

FEATURES = [
    'Max Packet Length',
    'Average Packet Size',
    'Fwd Packet Length Min',
    'Min Packet Length',
    'Fwd Packet Length Max',
    'Packet Length Mean',
    'Fwd Packet Length Mean',
    'Avg Fwd Segment Size',
    'min_seg_size_forward',
    'ACK Flag Count',
    'Flow Duration',
    'Fwd IAT Total',
    'Flow IAT Max',
    'Fwd IAT Max',
    'Flow IAT Std',
    'Fwd IAT Std',
    'Fwd IAT Mean',
    'Flow IAT Mean',
    'Total Length of Fwd Packets',
    'Subflow Fwd Bytes',
    'act_data_pkt_fwd',
    'Subflow Fwd Packets',
    'Total Fwd Packets',
    'Down/Up Ratio',
    'Init_Win_bytes_backward',
    'Total Length of Bwd Packets',
    'Subflow Bwd Bytes',
    'Flow IAT Min',
    'Bwd Packet Length Max',
    'URG Flag Count',
    'Bwd IAT Total',
    'Bwd Packets/s',
    'Init_Win_bytes_forward',
]

## Loop de treinamento default

In [ ]:
%load_ext autoreload
%autoreload 2

from datetime import datetime
import pandas as pd

default_params = {
    'AIF': {
        'window_size': 256,
        'n_trees': 100,
        'height': None,
        'm_trees': 10,
        'weights': 0.5
    },
    'HST': {
        'window_size': 250,
        'number_of_trees': 25,
        'max_depth': 15,
        'anomaly_threshold': 0.50,
        'size_limit': 0.10
    },
    'AE': {
        'hidden_layer': 2,
        'learning_rate': 0.5,
        'threshold': 0.60
    }
}

modelos_para_executar = ['AE', 'HST', 'AIF']
estrategias_discretizacao = ['z_score', 'dinamic']

configuracoes_features = [
    ('FullFeatures', None),
    ('33Features', FEATURES)
]

id_execucao = datetime.now().strftime("%Y%m%d_%H%M")

for dataset_path in datasets:
    nome_experimento = dataset_path.split('/')[-1].replace('.csv', '')
    
    df = pd.read_csv(dataset_path)
    
    for feat_name, selected_feats in configuracoes_features:
        
        processor = DataStreamProcessor(logging=False, selected_features=selected_feats)
        
        stream, targets, features = processor.create_stream(
            df=df,
            target_label_col='Label',
            binary_label=False,
            normalize_method="MinMaxScaler",
            threshold_var=None,
            threshold_corr=None,
            top_n_features=None,
            return_stream=True,
            extra_ignore_cols=['Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Protocol', 'Inbound'],
            imputation_method='mediana'
        )
        
        for estrategia in estrategias_discretizacao:
            for model_name in modelos_para_executar:
                print(f"\n---> Avaliando | Modelo: {model_name} | Discrtz: {estrategia.upper()} | Feats: {feat_name} <---")

                current_params = default_params[model_name].copy()
                
                if estrategia == 'z_score':
                    current_params['z'] = 3.0
                    discretization_val = 0.5
                elif estrategia == 'dinamic':
                    discretization_val = current_params.get('threshold', current_params.get('anomaly_threshold', 0.50))
                else:
                    discretization_val = 0.5
                
                model_init_params = current_params.copy()
                model_init_params.pop('z', None)
                
                kwargs = {f"{model_name.lower()}_params": model_init_params}
                algoritmos = get_anomaly_models(
                    stream.get_schema(),
                    selected_models=[model_name],
                    **kwargs
                )

                runner = AnomalyExperimentRunner(target_names=targets, n_runs=5)
    
                runner.run_anomaly_evaluation(
                    stream=stream,
                    algorithms=algoritmos,
                    window_evaluation=100,
                    warmup_instances=2000,
                    title=nome_experimento,
                    discretization=discretization_val,
                    algorithm_params=current_params,
                    is_optimized=False,
                    num_features=len(features),
                    exec_id=id_execucao
                )

## Adaptative Isolation Forest (AIF)

### Todas as características

In [ ]:
%load_ext autoreload
%autoreload 2

from datetime import datetime

default_aif = {
    'window_size': 256,
    'n_trees': 100,
    'height': None,
    'm_trees': 10,
    'weights': 0.5
}

id_execucao = datetime.now().strftime("%Y%m%d_%H%M")

for dataset_path in datasets:
    nome_experimento = dataset_path.split('/')[-1].replace('.csv', '')
    print(f"\nIniciando treinamento para: {nome_experimento}")
    
    df = pd.read_csv(dataset_path)
    
    processor = DataStreamProcessor(logging=False)
    
    stream, targets, features = processor.create_stream(
        df=df,
        target_label_col='Label',
        binary_label=False,
        normalize_method="MinMaxScaler",
        threshold_var=None,
        threshold_corr=None,
        top_n_features=None,
        return_stream=True,
        extra_ignore_cols=['Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Protocol', 'Inbound'],
        imputation_method='mediana'
    )
    
    algoritmos = get_anomaly_models(
        stream.get_schema(),
        selected_models=['AIF'],
        aif_params=default_aif
    )
    
    runner = AnomalyExperimentRunner(target_names=targets, n_runs=5)
    
    runner.run_anomaly_evaluation(
        stream,
        algorithms=algoritmos,
        window_evaluation=100,
        warmup_instances=0,
        title=nome_experimento,
        discretization=0.50,
        algorithm_params=default_aif,          
        is_optimized=False,                       
        num_features=len(features),
        exec_id=id_execucao
    )

### Melhores Características

In [ ]:
%load_ext autoreload
%autoreload 2

from datetime import datetime

default_aif = {
    'window_size': 256,
    'n_trees': 100,
    'height': None,
    'm_trees': 10,
    'weights': 0.5
}

id_execucao = datetime.now().strftime("%Y%m%d_%H%M")

for dataset_path in datasets:
    nome_experimento = dataset_path.split('/')[-1].replace('.csv', '')
    print(f"\nIniciando treinamento para: {nome_experimento}")
    
    df = pd.read_csv(dataset_path)
    
    processor = DataStreamProcessor(logging=False, selected_features=FEATURES)
    
    stream, targets, features = processor.create_stream(
        df=df,
        target_label_col='Label',
        binary_label=False,
        normalize_method="MinMaxScaler",
        threshold_var=None,
        threshold_corr=None,
        top_n_features=None,
        return_stream=True,
        extra_ignore_cols=['Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Protocol', 'Inbound'],
        imputation_method='mediana'
    )
    
    algoritmos = get_anomaly_models(
        stream.get_schema(),
        selected_models=['AIF'],
        aif_params=default_aif
    )
    
    runner = AnomalyExperimentRunner(target_names=targets, n_runs=5)
    
    runner.run_anomaly_evaluation(
        stream,
        algorithms=algoritmos,
        window_evaluation=100,
        warmup_instances=0,
        title=nome_experimento,
        discretization=0.50,
        algorithm_params=default_aif,          
        is_optimized=False,                       
        num_features=len(features),
        exec_id=id_execucao
    )

### Tabela AIF

A tabela pega dos arquivos CSVs gerados para construção do PNG e da tabela formatada para overleaf

In [ ]:
%load_ext autoreload
%autoreload 2

from src.Results.Plots import Plots

# Especifique os caminhos dos 4 arquivos CSV do algoritmo que deseja resumir
csvs_aif = {
    "output/AdaptiveIsolationForest/AdaptiveIsolationForest_Otimizado_FullFeatures.csv": "20260416_1737",
    "output/AdaptiveIsolationForest/AdaptiveIsolationForest_Otimizado_33Features.csv": "20260416_1943",
    "output/AdaptiveIsolationForest/AdaptiveIsolationForest_Default_FullFeatures.csv": "20260416_1901",
    "output/AdaptiveIsolationForest/AdaptiveIsolationForest_Default_33Features.csv": "20260416_1943",
}

# Inicializamos a classe Plots
plots = Plots(target_names=['Normal', 'Ataque'])

plots.plot_summary_table(
    csv_dict=csvs_aif, 
    algo_name="AdaptiveIsolationForest",
    exclude_blocks=['25'],
    include_blocks=['200', '1000']
)

## Half Space Trees (HST)

### Todas as Características

In [ ]:
%load_ext autoreload
%autoreload 2

from datetime import datetime

default_hst = {
    'window_size': 250,
    'number_of_trees': 25,
    'max_depth': 15,
    'anomaly_threshold': 0.50,
    'size_limit': 0.10
}

id_execucao = datetime.now().strftime("%Y%m%d_%H%M")

for dataset_path in datasets:
    nome_experimento = dataset_path.split('/')[-1].replace('.csv', '')
    print(f"\nIniciando treinamento para: {nome_experimento}")
    
    df = pd.read_csv(dataset_path)
    
    processor = DataStreamProcessor(logging=False)
    
    stream, targets, features = processor.create_stream(
        df=df,
        target_label_col='Label',
        binary_label=False,
        normalize_method="MinMaxScaler",
        threshold_var=None,
        threshold_corr=None,
        top_n_features=None,
        return_stream=True,
        extra_ignore_cols=['Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Protocol', 'Inbound'],
        imputation_method='mediana'
    )
    
    algoritmos = get_anomaly_models(
        stream.get_schema(),
        selected_models=['HST'],
        hst_params=default_hst
    )
    
    runner = AnomalyExperimentRunner(target_names=targets, n_runs=5)
    
    runner.run_anomaly_evaluation(
        stream,
        algorithms=algoritmos,
        window_evaluation=100,
        warmup_instances=0,
        title=nome_experimento,
        discretization=0.50,
        algorithm_params=default_hst,          
        is_optimized=False,                       
        num_features=len(features),
        exec_id=id_execucao
    )

### Melhores características

In [ ]:
%load_ext autoreload
%autoreload 2

from datetime import datetime

default_hst = {
    'window_size': 250,
    'number_of_trees': 25,
    'max_depth': 15,
    'anomaly_threshold': 0.50,
    'size_limit': 0.10
}

id_execucao = datetime.now().strftime("%Y%m%d_%H%M")

for dataset_path in datasets:
    nome_experimento = dataset_path.split('/')[-1].replace('.csv', '')
    print(f"\nIniciando treinamento para: {nome_experimento}")
    
    df = pd.read_csv(dataset_path)
    
    processor = DataStreamProcessor(logging=False, selected_features=FEATURES)
    
    stream, targets, features = processor.create_stream(
        df=df,
        target_label_col='Label',
        binary_label=False,
        normalize_method="MinMaxScaler",
        threshold_var=None,
        threshold_corr=None,
        top_n_features=None,
        return_stream=True,
        extra_ignore_cols=['Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Protocol', 'Inbound'],
        imputation_method='mediana'
    )
    
    algoritmos = get_anomaly_models(
        stream.get_schema(),
        selected_models=['HST'],
        hst_params=default_hst
    )
    
    runner = AnomalyExperimentRunner(target_names=targets, n_runs=5)
    
    runner.run_anomaly_evaluation(
        stream,
        algorithms=algoritmos,
        window_evaluation=100,
        warmup_instances=0,
        title=nome_experimento,
        discretization=0.50,
        algorithm_params=default_hst,          
        is_optimized=False,                       
        num_features=len(features),
        exec_id=id_execucao
    )

### Tabela HST

In [ ]:
%load_ext autoreload
%autoreload 2

from src.Results.Plots import Plots

# Especifique os caminhos dos 4 arquivos CSV do algoritmo que deseja resumir
csvs_aif = {
    "output/HalfSpaceTrees/HalfSpaceTrees_Otimizado_FullFeatures.csv": "20260416_2224",
    "output/HalfSpaceTrees/HalfSpaceTrees_Otimizado_33Features.csv": "20260416_2224",
    "output/HalfSpaceTrees/HalfSpaceTrees_Default_FullFeatures.csv": "20260417_1323",
    "output/HalfSpaceTrees/HalfSpaceTrees_Default_33Features.csv": "20260417_1325",
}

# Inicializamos a classe Plots
plots = Plots(target_names=['Normal', 'Ataque'])

plots.plot_summary_table(
    csv_dict=csvs_aif, 
    algo_name="HalfSpaceTrees",
    exclude_blocks=['25'],
    include_blocks=['200', '1000']
)

## Autoencoder (AE)

### Todas as características

In [ ]:
%load_ext autoreload
%autoreload 2

from datetime import datetime

default_ae = {
    'hidden_layer': 2,
    'learning_rate': 0.5,
    'threshold': 0.60
}

id_execucao = datetime.now().strftime("%Y%m%d_%H%M")

for dataset_path in datasets:
    nome_experimento = dataset_path.split('/')[-1].replace('.csv', '')
    print(f"\nIniciando treinamento para: {nome_experimento}")
    
    df = pd.read_csv(dataset_path)
    
    processor = DataStreamProcessor(logging=False)
    
    stream, targets, features = processor.create_stream(
        df=df,
        target_label_col='Label',
        binary_label=False,
        normalize_method="MinMaxScaler",
        threshold_var=None,
        threshold_corr=None,
        top_n_features=None,
        return_stream=True,
        extra_ignore_cols=['Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Protocol', 'Inbound'],
        imputation_method='mediana'
    )
    
    algoritmos = get_anomaly_models(
        stream.get_schema(),
        selected_models=['AE'],
        ae_params=default_ae
    )
    
    runner = AnomalyExperimentRunner(target_names=targets, n_runs=5)
    
    runner.run_anomaly_evaluation(
        stream,
        algorithms=algoritmos,
        window_evaluation=100,
        warmup_instances=0,
        title=nome_experimento,
        discretization=0.60,
        algorithm_params=default_ae,          
        is_optimized=False,                       
        num_features=len(features),
        exec_id=id_execucao
    )

### Melhores características

In [ ]:
%load_ext autoreload
%autoreload 2

from datetime import datetime

default_ae = {
    'hidden_layer': 2,
    'learning_rate': 0.5,
    'threshold': 0.60
}

id_execucao = datetime.now().strftime("%Y%m%d_%H%M")

for dataset_path in datasets:
    nome_experimento = dataset_path.split('/')[-1].replace('.csv', '')
    print(f"\nIniciando treinamento para: {nome_experimento}")
    
    df = pd.read_csv(dataset_path)
    
    processor = DataStreamProcessor(logging=False, selected_features=FEATURES)
    
    stream, targets, features = processor.create_stream(
        df=df,
        target_label_col='Label',
        binary_label=False,
        normalize_method="MinMaxScaler",
        threshold_var=None,
        threshold_corr=None,
        top_n_features=None,
        return_stream=True,
        extra_ignore_cols=['Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Protocol', 'Inbound'],
        imputation_method='mediana'
    )
    
    algoritmos = get_anomaly_models(
        stream.get_schema(),
        selected_models=['AE'],
        ae_params=default_ae
    )
    
    runner = AnomalyExperimentRunner(target_names=targets, n_runs=5)
    
    runner.run_anomaly_evaluation(
        stream,
        algorithms=algoritmos,
        window_evaluation=100,
        warmup_instances=1000,
        title=nome_experimento,
        discretization=0.855,
        algorithm_params=default_ae,          
        is_optimized=False,                       
        num_features=len(features),
        exec_id=id_execucao
    )

### Tabela AE

In [ ]:
%load_ext autoreload
%autoreload 2

from src.Results.Plots import Plots

# Especifique os caminhos dos 4 arquivos CSV do algoritmo que deseja resumir
csvs_aif = {
    "output/Autoencoder/Autoencoder_Otimizado_FullFeatures.csv": "20260417_1904",
    "output/Autoencoder/Autoencoder_Otimizado_33Features.csv": "20260417_1548",
    "output/Autoencoder/Autoencoder_Default_FullFeatures.csv": "20260417_1533",
    "output/Autoencoder/Autoencoder_Default_33Features.csv": "20260417_1548",
}

# Inicializamos a classe Plots
plots = Plots(target_names=['Normal', 'Ataque'])

plots.plot_summary_table(
    csv_dict=csvs_aif, 
    algo_name="Autoencoder",
    exclude_blocks=['25'],
    include_blocks=['200', '1000']
)